### Importação das bibliotecas

In [80]:
import pandas as pd
import re
from unidecode import unidecode

### Carregamento dos dados

In [81]:
df = pd.read_json("../data/raw/clientes_crm.json").set_index("code")
df.head(10)

,full_name,location,email
code,,,
1,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",femininos.oliveira.antunes@icloud.com
2,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",nunes.fernanda.soares.azevedo.vieira@outlook.com
3,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",farias.teixeira.daniel.ribeiro#gmail.com
4,Thiago Moreira,"AC , Rio Branco",thiago.moreira#gmail.com
5,Pedro Freitas,PA - Santarém Novo,pedro.freitas#icloud.com
6,Antônia Coelho Pinheiro Peixoto Cavalcanti,"Fortaleza do Tabocão , TO",coelho.pinheiro.peixoto.antônia.cavalcanti@aol...
7,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo,torres.barros.rocha.bianca.siqueira#aol.com
8,Luiz Alves Pimentel,SE - Aracaju,pimentel.alves.luiz#outlook.com
9,Lucas Guedes Cunha Lopes,PB - João Pessoa,lucas.lopes.guedes.cunha#tutanota.com


primeira observação é o email, possui acentos e em alguns, no lugar do @ esta um #

In [82]:
df['email'] = df['email'].apply(unidecode).str.replace("#", "@")

df.head(10)

,full_name,location,email
code,,,
1,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",femininos.oliveira.antunes@icloud.com
2,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",nunes.fernanda.soares.azevedo.vieira@outlook.com
3,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",farias.teixeira.daniel.ribeiro@gmail.com
4,Thiago Moreira,"AC , Rio Branco",thiago.moreira@gmail.com
5,Pedro Freitas,PA - Santarém Novo,pedro.freitas@icloud.com
6,Antônia Coelho Pinheiro Peixoto Cavalcanti,"Fortaleza do Tabocão , TO",coelho.pinheiro.peixoto.antonia.cavalcanti@aol...
7,Bianca Barros Rocha Torres Siqueira,PB/Cabedelo,torres.barros.rocha.bianca.siqueira@aol.com
8,Luiz Alves Pimentel,SE - Aracaju,pimentel.alves.luiz@outlook.com
9,Lucas Guedes Cunha Lopes,PB - João Pessoa,lucas.lopes.guedes.cunha@tutanota.com


Outro problema é a coluna "location", nela temos um monte de possibilidades com estado e cidade invertendo de posição e separador, irei padronizar isso!

In [83]:
def tratar_location(texto):
    # 1. Remove espaços extras
    texto = texto.strip()
    
    # 2. Padroniza separadores para |
    texto = re.sub(r'\s*[,\-/]\s*', '|', texto)
    
    # 3. Separa as partes
    partes = [parte.strip() for parte in texto.split('|')]
    
    # 4. Identifica qual parte é o estado (2 letras maiúsculas)
    estado = None
    cidade = None
    for parte in partes:
        if re.match(r'^[A-Z]{2}$', parte):
            estado = parte
        else:
            cidade = parte
    
    return pd.Series({'cidade': cidade, 'estado': estado})

In [84]:
df[['cidade', 'estado']] = df['location'].apply(tratar_location)
df = df.drop(columns=['location'])

In [85]:
df.head(10)

,full_name,email,cidade,estado
code,,,,
1,Femininos Oliveira Antunes,femininos.oliveira.antunes@icloud.com,Aratu (Candeias),BA
2,Fernanda Azevedo Soares Nunes Vieira,nunes.fernanda.soares.azevedo.vieira@outlook.com,Recife,PE
3,Daniel Farias Ribeiro Teixeira,farias.teixeira.daniel.ribeiro@gmail.com,Rio Grande,RS
4,Thiago Moreira,thiago.moreira@gmail.com,Rio Branco,AC
5,Pedro Freitas,pedro.freitas@icloud.com,Santarém Novo,PA
6,Antônia Coelho Pinheiro Peixoto Cavalcanti,coelho.pinheiro.peixoto.antonia.cavalcanti@aol...,Fortaleza do Tabocão,TO
7,Bianca Barros Rocha Torres Siqueira,torres.barros.rocha.bianca.siqueira@aol.com,Cabedelo,PB
8,Luiz Alves Pimentel,pimentel.alves.luiz@outlook.com,Aracaju,SE
9,Lucas Guedes Cunha Lopes,lucas.lopes.guedes.cunha@tutanota.com,João Pessoa,PB


In [86]:
df.isna().sum()

full_name    0
email        0
cidade       0
estado       0
dtype: int64

In [87]:
df.duplicated().sum()

np.int64(0)

In [88]:
df.reset_index(drop=True)

,full_name,email,cidade,estado
0,Femininos Oliveira Antunes,femininos.oliveira.antunes@icloud.com,Aratu (Candeias),BA
1,Fernanda Azevedo Soares Nunes Vieira,nunes.fernanda.soares.azevedo.vieira@outlook.com,Recife,PE
2,Daniel Farias Ribeiro Teixeira,farias.teixeira.daniel.ribeiro@gmail.com,Rio Grande,RS
3,Thiago Moreira,thiago.moreira@gmail.com,Rio Branco,AC
4,Pedro Freitas,pedro.freitas@icloud.com,Santarém Novo,PA
5,Antônia Coelho Pinheiro Peixoto Cavalcanti,coelho.pinheiro.peixoto.antonia.cavalcanti@aol...,Fortaleza do Tabocão,TO
6,Bianca Barros Rocha Torres Siqueira,torres.barros.rocha.bianca.siqueira@aol.com,Cabedelo,PB
7,Luiz Alves Pimentel,pimentel.alves.luiz@outlook.com,Aracaju,SE
8,Lucas Guedes Cunha Lopes,lucas.lopes.guedes.cunha@tutanota.com,João Pessoa,PB
9,Débora Paiva,paiva.debora@gmx.com,Santarém,PA


Agora está tudo certo!

In [89]:
df.to_csv('../data/processed/clientes_crm_processed.csv', index=False)